# 3. Challenge the stream, then evaluate a frozen configuration
 **Question:** does the controller make its failure behavior visible?
 Development fault checks use R08. A separate final cell uses R12 only after choices are frozen. No notebook outputs are precomputed.

In [ ]:
%pip -q install "numpy>=1.26,<3" "scipy>=1.11,<2" "mne>=1.8,<2" "scikit-learn>=1.4,<2" "matplotlib>=3.8,<4"

In [ ]:
from pathlib import Path
import hashlib, sys, urllib.request
SOURCE_COMMIT = "22dcdced86cc425a2d50ffa8516fe9581730ebc9"
EXPECTED_SHA256 = "90c8c015e114b3b9a89bb4206dbc4f6094969a792b1a0263e0009e673f935567"
candidates = [Path("bci_studio.py"), Path("chapters/ch20-bci-foundations/bci_studio.py")]
helper = next((p for p in candidates if p.is_file()), None)
if helper is None:
    target = Path("chapter20_runtime")
    target.mkdir(exist_ok=True)
    helper = target / "bci_studio.py"
    url = f"https://raw.githubusercontent.com/ZagElSayed/Introduction-to-Neuroinformatics/{SOURCE_COMMIT}/chapters/ch20-bci-foundations/bci_studio.py"
    payload = urllib.request.urlopen(url, timeout=60).read()
    if hashlib.sha256(payload).hexdigest() != EXPECTED_SHA256:
        raise RuntimeError("Downloaded helper checksum mismatch")
    helper.write_bytes(payload)
sys.path.insert(0, str(helper.resolve().parent))
print("Helper:", helper, "SHA-256:", hashlib.sha256(helper.read_bytes()).hexdigest())
import bci_studio as bci

In [ ]:
experiment = bci.build_experiment()
clean = bci.replay(experiment)
faulted = bci.replay(experiment, faults=True)
print("Clean:", clean["summary"])
print("Faulted:", faulted["summary"])
print("Events:", faulted["events"])

## Fault contract
 At recorded time 20 s, drop one 40-sample chunk; the next packet exposes the sample-index gap. At 30 s, delay a chunk by .75 s in the virtual arrival clock, beyond the .50-s age limit. At 40 s, flatten C3 for one chunk.
 On each detected problem the pipeline discards the chunk, resets filter/buffer/evidence, emits no command, and requires a fresh two-second window before predicting again. A virtual delay also stalls arrival of queued packets, keeping arrival timestamps monotonic.
 The limits are illustrative engineering settings, not validated EEG artifact criteria. There is no arbitrary write, stimulation, device actuation, or live network replay.

In [ ]:
invalid = {"sample_gap", "stale_packet", "flat_channel", "amplitude_limit", "nonfinite", "warming_up"}
assert all(r["command"] == r["naive_command"] == 0 for r in faulted["rows"] if r["status"] in invalid)
assert all(b["arrival_time"] >= a["arrival_time"] for a, b in zip(faulted["rows"], faulted["rows"][1:]))
print("Student-run checks passed for inhibition and virtual clock order.")
print(bci.save_bundle(faulted))

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(11, 3))
ax.step([r["sample_time"] for r in faulted["rows"]], [r["cursor"] for r in faulted["rows"]], where="post", color="#167D91")
for event in faulted["events"]:
    ax.axvline(event["sample"] / experiment["config"].fs, color="#D65473", alpha=.5)
ax.set(xlabel="Recorded sample time (s)", ylabel="Virtual position", title="Fault events and the gated cursor")
plt.show()

## Final held-out run: freeze first
 Set `FROZEN = True` only after writing down the chosen features, threshold, dwell, QC, and recovery rules using development evidence. R12 is a held-out **run**, not evidence of participant or session transfer. If you revise after seeing it, treat it as development evidence and obtain a new test.

In [ ]:
FROZEN = False
if not FROZEN:
    raise RuntimeError("Freeze the configuration and document the decision before opening R12.")
offline_test = bci.held_out_metrics(experiment)
final_replay = bci.replay(experiment, run=experiment["config"].test_run)
print("One-window-per-trial offline test:", offline_test)
print("Past-window continuous replay:", final_replay["summary"])
print(bci.save_bundle({"kind": "final_evaluation", "offline_test": offline_test, "replay": final_replay}))

## Submit the evidence, not an impressive score
 Report trial counts, balanced accuracy/confusion, task-window and conditional-command accuracy, accepted command counts, rest proxy rate and exposure, rejected/warm-up intervals, fault events, and processing median/p95.
 Offline trial and continuous window metrics have different units of analysis; do not compare them as interchangeable accuracies. No ITR is estimated from the overlapping replay windows.
 Identify limits: one participant, cued rest, simple QC, uncalibrated scores, overlapping windows, simulated clock, no physical feedback or online learning. Follow **ASSIGNMENT.md** and **REFLECTION.md**.